In [ ]:
# instalação das bibliotecas
!pip install pandas numpy matplotlib scipy scikit-learn

In [ ]:
# importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [ ]:
# leitura dos dados
dados = pd.read_csv("servidores_ti.csv")
print(f"Dataset original: {dados.shape}")

dados.head()

Dataset original: (500, 8)


In [ ]:
# tipos de dados de cada coluna
print(dados.dtypes)

In [ ]:
# estatísticas descritivas de uso_cpu e latencia_rede_ms
# ddof=1 porque os dados são tratados como amostra, não como a população inteira
cols = ["uso_cpu", "latencia_rede_ms"]

resumo = pd.DataFrame({
    "media": dados[cols].mean(),
    "variancia": dados[cols].var(ddof=1),
    "desvio_padrao": dados[cols].std(ddof=1),
    "n": dados[cols].count()
})
resumo

In [ ]:
# intervalo de confiança de 95% para a média de uso_cpu
# usa a distribuição t de Student, pois o desvio padrão da população é desconhecido
n = dados["uso_cpu"].count()
cpu_med = dados["uso_cpu"].mean()
cpu_dp = dados["uso_cpu"].std(ddof=1)
erro_pad = cpu_dp / np.sqrt(n)
conf = 0.95
alfa = 1 - conf
t_crit = stats.t.ppf(1 - alfa/2, df=n-1)
margem = t_crit * erro_pad

lim_inf = cpu_med - margem
lim_sup = cpu_med + margem

print(f"n = {n}")
print(f"Média amostral = {cpu_med:.3f}%")
print(f"Desvio padrão amostral = {cpu_dp:.3f}")
print(f"Erro padrão (s/√n) = {erro_pad:.4f}")
print(f"t crítico (gl={n-1}, 95%) = {t_crit:.4f}")
print(f"Margem de erro = {margem:.4f}")
print(f"IC 95% para a média de uso_cpu: [{lim_inf:.2f}% , {lim_sup:.2f}%]")

## Parte 2 -- tipo de rede tem relação com os alertas de falha?

In [ ]:
# tabela cruzada entre tipo de rede e status de alerta
cruz = pd.crosstab(dados["tipo_rede"], dados["status_alerta"],
                    rownames=["tipo_rede"], colnames=["status_alerta"])

cruz.columns = ["Saudável (0)", "Falha Iminente (1)"]
cruz

In [ ]:
# teste qui-quadrado de independência
# H0: tipo de rede e status de alerta são independentes
qui2, p_val, gl, freq_esp = stats.chi2_contingency(cruz)

print(f"Estatística Qui-Quadrado (χ²) = {qui2:.4f}")
print(f"Graus de liberdade = {gl}")
print(f"p-valor = {p_val:.4f}")
print("\nFrequências esperadas sob H0 (independência):")
print(pd.DataFrame(freq_esp,
                    index=cruz.index,
                    columns=cruz.columns).round(2))


alfa = 0.05
if p_val < alfa:
    print(f"\nComo p-valor ({p_val:.4f}) < α ({alfa}), REJEITAMOS H0.")
else:
    print(f"\nComo p-valor ({p_val:.4f}) >= α ({alfa}), NÃO rejeitamos H0.")
# rejeitar H0 indica que tipo de rede e status de alerta não são independentes

## Parte 3 -- reduzindo as variáveis com PCA

In [ ]:
# divisão entre treino e teste (antes da padronização e do PCA, para que
# o ajuste dessas etapas não use informação dos dados de teste)
cols_num = ["uso_cpu", "uso_memoria", "latencia_rede_ms", "taxa_pacotes_perdidos"]

X = dados[cols_num].values
y = dados["status_alerta"].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Tamanho do treino: {X_tr.shape[0]} amostras")
print(f"Tamanho do teste:  {X_te.shape[0]} amostras")
print(f"Proporção de falhas (classe 1) no treino: {y_tr.mean()*100:.1f}%")
print(f"Proporção de falhas (classe 1) no teste:  {y_te.mean()*100:.1f}%")

In [ ]:
# padronização das variáveis (média 0 e desvio padrão 1)
# scaler ajustado só com os dados de treino, e aplicado no treino e no teste
norm = StandardScaler()
X_tr_pad = norm.fit_transform(X_tr)
X_te_pad = norm.transform(X_te)
print("Média no treino após padronização (deve ser ~0):", X_tr_pad.mean(axis=0).round(3))
print("Desvio padrão no treino após padronização (deve ser ~1):", X_tr_pad.std(axis=0).round(3))

In [ ]:
# PCA reduzindo as 4 variáveis originais para 2 componentes principais
# ajustado só no treino, e aplicado no treino e no teste
pca = PCA(n_components=2)
X_tr_pca = pca.fit_transform(X_tr_pad)
X_te_pca = pca.transform(X_te_pad)

var_exp = pca.explained_variance_ratio_
var_ac = np.cumsum(var_exp)

print("Variância explicada por componente:")
for i, (v, vac) in enumerate(zip(var_exp, var_ac), start=1):
    print(f"  PC{i}: {v*100:.2f}% (acumulada: {vac*100:.2f}%)")

In [ ]:
# peso de cada variável original em PC1 e PC2
cargas = pd.DataFrame(pca.components_, columns=cols_num, index=["PC1", "PC2"])
cargas.round(3)

## Parte 4 -- treinando um classificador (KNN) pra prever falha

In [ ]:
# treino do KNN com k=5
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_tr_pca, y_tr)
y_prev = knn.predict(X_te_pca)

In [ ]:
# matriz de confusão
matriz = confusion_matrix(y_te, y_prev)


fig, ax = plt.subplots(figsize=(5.5, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=matriz,
                               display_labels=["Saudável (0)", "Falha Iminente (1)"])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Matriz de Confusão — KNN (K=5)")
plt.tight_layout()
plt.show()

print(matriz)

In [ ]:
# relatório de classificação (precisão, recall e f1-score por classe)
# atenção principalmente na classe "Falha Iminente", que é a de interesse
relat = classification_report(y_te, y_prev,
                               target_names=["Saudável (0)", "Falha Iminente (1)"],
                               digits=3)
print(relat)